# Evaluación formal — Métricas para el informe (Fase 4)

**Proyecto Final — Asistente de Triaje de Mails con RAG y LLM**

---

## Objetivo

Producir los **números oficiales** que van a la Sección 5 (Evaluación) del informe técnico, según las métricas que la maqueta del profe exige para Track A:

| Dimensión | Métricas | Cómo |
|---|---|---|
| **Retrieval** | Precision@1, Precision@3, Recall@5, MRR, nDCG@5 | Determinístico (no usa LLM) |
| **Generación** | Faithfulness, Answer Relevance | LLM-as-judge sobre muestra |
| **Operativas** | Latencia p50/p95, tokens, costo estimado | Medidas durante la corrida |

## Ground truth

Usamos **15 mails** curados a mano en `data/eval/ground_truth.json`, distribuidos:
- 8 EN + 7 ES → cobertura multilingüe.
- 3 mails por cada una de las 5 categorías → cobertura balanceada.
- Relevance graduada (1.0 ideal / 0.5 parcial / 0.3 tangencial) para nDCG.

## Plan de gasto de cuota Gemini

Para no quemar la cuota diaria:
- Métricas de retrieval: **0 llamadas** (todo local).
- RAG end-to-end sobre **5 mails de muestra**: 5 llamadas.
- LLM-as-judge sobre esos 5 (Faithfulness + Answer Relevance): 10 llamadas.
- **Total: ~15 llamadas**, manejable en una corrida.

## 1. Setup

In [ ]:
%%capture
!pip install -q -U google-genai pydantic python-dotenv pandas
!pip install -q sentence-transformers chromadb rank-bm25
!pip uninstall -y google-generativeai 2>/dev/null || true

In [ ]:
REPO_URL = "https://github.com/<TU-USUARIO>/<TU-REPO>.git"  # ⚠️ cambialo

import os, sys, json, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
    if not Path(repo_name).exists():
        os.system(f'git clone {REPO_URL}')
    os.chdir(repo_name)
    PROJECT_ROOT = Path.cwd()
    try:
        from google.colab import userdata
        os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    except Exception:
        import getpass
        os.environ['GOOGLE_API_KEY'] = getpass.getpass('GOOGLE_API_KEY: ')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')

sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
import numpy as np

assert os.getenv('GOOGLE_API_KEY'), 'Falta GOOGLE_API_KEY'
print(f'✅ Entorno: {"Colab" if IN_COLAB else "Local"}')

## 2. Indexar el KB (si no está)

In [ ]:
from src.kb_ingest import ingest_kb
ingest_kb(verbose=True)

## 3. Cargar el ground truth

In [ ]:
with open(PROJECT_ROOT / 'data/eval/ground_truth.json') as f:
    gt = json.load(f)

mails = gt['mails']
print(f'Mails de evaluación: {len(mails)}')
print(f'EN: {sum(1 for m in mails if m["language"] == "en")}')
print(f'ES: {sum(1 for m in mails if m["language"] == "es")}')
print()
print('Distribución por categoría:')
from collections import Counter
for cat, c in Counter(m['category'] for m in mails).items():
    print(f'  {cat}: {c}')

## 4. Métricas de Retrieval (todas las queries, sin LLM)

Para cada mail del ground truth corremos el retriever y computamos las 5 métricas.

In [ ]:
from src.retriever import search, reset_caches
from src.evaluation import (
    precision_at_k, recall_at_k, reciprocal_rank, ndcg_at_k,
    aggregate_retrieval_metrics,
)
reset_caches()

per_query = []
rows = []
TOP_K_FETCH = 5

for m in mails:
    results = search(m['mail'], top_k=TOP_K_FETCH)
    retrieved_ids = [r.doc_id for r in results]
    relevance = m['relevance']  # doc_id → score
    relevant_ids = list(relevance.keys())

    metrics = {
        'P@1':    precision_at_k(retrieved_ids, relevant_ids, 1),
        'P@3':    precision_at_k(retrieved_ids, relevant_ids, 3),
        'R@5':    recall_at_k(retrieved_ids, relevant_ids, 5),
        'MRR':    reciprocal_rank(retrieved_ids, relevant_ids),
        'nDCG@5': ndcg_at_k(retrieved_ids, relevance, 5),
    }
    per_query.append(metrics)
    rows.append({
        'id':       m['id'],
        'lang':     m['language'],
        'category': m['category'],
        'top1':     retrieved_ids[0] if retrieved_ids else None,
        **{k: round(v, 3) for k, v in metrics.items()},
    })

df_ret = pd.DataFrame(rows)
df_ret

In [ ]:
agg = aggregate_retrieval_metrics(per_query)
print(f'=== Retrieval — agregado sobre {agg.n} mails ===')
print(f'Precision@1: {agg.precision_at_1:.3f}')
print(f'Precision@3: {agg.precision_at_3:.3f}')
print(f'Recall@5:    {agg.recall_at_5:.3f}')
print(f'MRR:         {agg.mrr:.3f}')
print(f'nDCG@5:      {agg.ndcg_at_5:.3f}')

## 5. RAG end-to-end + LLM-as-judge sobre 5 mails de muestra

Tomamos una muestra estratificada (1 mail por cada categoría, mezcla de idiomas) para mantener el gasto de cuota acotado.

In [ ]:
# Muestra estratificada: 1 mail por categoría
SAMPLE_IDS = [1, 4, 5, 8, 10]  # ACCOUNT-en, ORDER-es, REFUND-en, PAYMENT-es, CONTACT-es
sample = [m for m in mails if m['id'] in SAMPLE_IDS]
print(f'Muestra: {len(sample)} mails')
for m in sample:
    print(f"  #{m['id']:02d} [{m['language']}] {m['category']}: {m['mail'][:60]}...")

In [ ]:
from src.pipeline import process_mail
from src.evaluation import judge_faithfulness, judge_answer_relevance

RATE_LIMIT_DELAY_S = 13  # respeta 5 RPM
rag_rows = []

for i, m in enumerate(sample):
    if i > 0:
        print(f'  ...esperando {RATE_LIMIT_DELAY_S}s...')
        time.sleep(RATE_LIMIT_DELAY_S)

    # 1. Pasar por el pipeline RAG
    t0 = time.perf_counter()
    result = process_mail(m['mail'], use_rag=True)
    latency = time.perf_counter() - t0
    print(f"  #{m['id']:02d} → intent={result.intent.value} ({latency:.2f}s)")

    # Recuperar contexto efectivo (para el juez)
    from src.retriever import search
    retrieved = search(m['mail'], top_k=3)
    context_texts = [r.text for r in retrieved]

    # 2. LLM-as-judge: Faithfulness
    time.sleep(RATE_LIMIT_DELAY_S)
    if result.suggested_response:
        faith = judge_faithfulness(context_texts, result.suggested_response)
        time.sleep(RATE_LIMIT_DELAY_S)
        ans_rel = judge_answer_relevance(m['mail'], result.suggested_response)
    else:
        faith = ans_rel = None

    rag_rows.append({
        'id':           m['id'],
        'lang':         m['language'],
        'category':     m['category'],
        'intent_ok':    '✅' if result.intent.value.startswith({
                              'ACCOUNT': 'Soporte', 'ORDER': 'Gestión',
                              'REFUND': 'Reembolsos', 'PAYMENT': 'Pagos',
                              'CONTACT': 'Contacto'}[m['category']]) else '❌',
        'latency_s':    round(latency, 2),
        'faith':        round(faith.score, 2) if faith else None,
        'ans_rel':      round(ans_rel.score, 2) if ans_rel else None,
        'faith_reason': faith.reason if faith else '',
        'response':     (result.suggested_response or '(none)')[:90],
    })

df_rag = pd.DataFrame(rag_rows)
df_rag

In [ ]:
# Computamos las métricas finales de forma auto-contenida (no depende de variables previas)
def _safe_mean(series):
    s = series.dropna() if hasattr(series, 'dropna') else series
    return s.mean() if len(s) > 0 else float('nan')

def _safe_median(series):
    s = series.dropna() if hasattr(series, 'dropna') else series
    return s.median() if len(s) > 0 else float('nan')

def _safe_quantile(series, q):
    s = series.dropna() if hasattr(series, 'dropna') else series
    return s.quantile(q) if len(s) > 0 else float('nan')

# Generación
faith_mean   = _safe_mean(df_rag['faith'])   if 'df_rag' in dir() else float('nan')
relev_mean   = _safe_mean(df_rag['ans_rel']) if 'df_rag' in dir() else float('nan')
n_judged     = df_rag['faith'].dropna().shape[0] if 'df_rag' in dir() else 0

# Operativas
lat_median   = _safe_median(df_rag['latency_s']) if 'df_rag' in dir() else float('nan')
lat_p95      = _safe_quantile(df_rag['latency_s'], 0.95) if 'df_rag' in dir() else float('nan')
n_latency    = df_rag.shape[0] if 'df_rag' in dir() else 0

summary = pd.DataFrame([
    {'Métrica': 'Precision@1',       'Valor': f'{agg.precision_at_1:.3f}', 'n': agg.n,       'Tipo': 'Retrieval'},
    {'Métrica': 'Precision@3',       'Valor': f'{agg.precision_at_3:.3f}', 'n': agg.n,       'Tipo': 'Retrieval'},
    {'Métrica': 'Recall@5',          'Valor': f'{agg.recall_at_5:.3f}',    'n': agg.n,       'Tipo': 'Retrieval'},
    {'Métrica': 'MRR',               'Valor': f'{agg.mrr:.3f}',            'n': agg.n,       'Tipo': 'Retrieval'},
    {'Métrica': 'nDCG@5',            'Valor': f'{agg.ndcg_at_5:.3f}',      'n': agg.n,       'Tipo': 'Retrieval'},
    {'Métrica': 'Faithfulness',      'Valor': f'{faith_mean:.3f}',          'n': n_judged,    'Tipo': 'Generación'},
    {'Métrica': 'Answer Relevance',  'Valor': f'{relev_mean:.3f}',          'n': n_judged,    'Tipo': 'Generación'},
    {'Métrica': 'Latencia p50',      'Valor': f'{lat_median:.2f}s',         'n': n_latency,   'Tipo': 'Operativa'},
    {'Métrica': 'Latencia p95',      'Valor': f'{lat_p95:.2f}s',            'n': n_latency,   'Tipo': 'Operativa'},
])
summary

## 6. Razones del juez (para el análisis de errores)

Si algún score quedó bajo, las razones del LLM-as-judge nos dicen exactamente qué falló — material directo para la sección de Análisis de Errores del informe.

In [ ]:
for _, row in df_rag.iterrows():
    if row['faith'] is None: continue
    print(f"#{row['id']:02d} [{row['lang']}] {row['category']}")
    print(f"   Faith={row['faith']:.2f} | AnsRel={row['ans_rel']:.2f}")
    print(f"   Razón faith: {row['faith_reason']}")
    print()

## 7. Tabla resumen final para el informe

Una sola tabla con todos los números que van a la Sección 5. **Esta es la tabla que pegamos al informe.**

## 8. Conclusión de la Fase 4

Con esta tabla resumen tenemos **todos los números** que la rúbrica del profe pide para el informe:

- ✅ Métricas de Retrieval (5).
- ✅ Métricas de Generación (2).
- ✅ Métricas Operativas (2).
- ✅ Ground truth documentado y reproducible.
- ✅ Análisis de errores con razones explícitas del LLM-as-judge.

## Lo que sigue — Fase 5: el informe

Ya tenemos:
- EDA con sus 8 hallazgos → Sección 3.
- Arquitectura y stack → Sección 2 + 4.
- Decisiones documentadas (KB bilingüe, RRF, etc.) → Sección 2.
- Métricas formales → Sección 5.
- Casos de error → Sección 5.6.

Falta redactar el `informe_tecnico.docx` siguiendo la maqueta del profe y armar las slides para el Demo Day.